# 北极星工具 HCCL-VM：安装与应用

前两章介绍了集合通信的概念、算子与算法。本章开始在**普通 CPU 环境**里用 HCCL-VM 验证通信任务。

本章的路径是：**认识原理 → 拿到环境 → 跑通第一个测试 → 配置自己的测试 → 组织一组用例 → 出问题时定位**。
环境部分走云端：下载一个 82 MB 的绿色包，点几下就出结果，**不用装 CANN、不用编译**。

> 想在自己电脑上长期折腾（改源码、改校验规则、重新打包），看 **3.9 本地 WSL 环境与自建绿色包（选看）**。

## 本章目录

| 小节 | 主题 | 打开 |
| :--- | :--- | :--- |
| **3.1 本章介绍** | 学习目标、全章目录与常用术语。 | 本文件 |
| **3.2 平台背景与原理** | 北极星的由来、分层架构，Runner 与 Checker 的分工，以及绿色包为什么能省掉装 CANN。 | [3.2](./03.02_hccl_vm_platform_overview.ipynb) |
| **3.3 获取并解压 HCCL-VM 绿色包** | 在 cannlab 环境里确认架构、下载、校验完整性、解压，并生成环境脚本。 | [3.3](./03.03_cloud_pack_setup.ipynb) |
| **3.4 认识测试用例：hccl_test 的用法与结果解读** | 测试用例负责什么、从哪来、有哪些、参数怎么给，以及模拟环境里为什么数值校验会失败。 | [3.4](./03.04_test_cases.ipynb) |
| **3.5 跑通第一个通信测试并解读结果** | 启动环境、建通信域、跑 ReduceScatter、用 Checker 验收，并导出结果。 | [3.5](./03.05_first_comm_test.ipynb) |
| **3.6 配置自己的通信测试：集群、通信域与 rank 映射** | 集群与通信域的关系、112/114/122 三种选择、rank 落在哪台模拟服务器。 | [3.6](./03.06_configure_tests.ipynb) |
| **3.7 通信用例实战：四个用例走完一组测试** | 改数据量、换算子、增加参与者、跨模拟服务器，四个用例一次跑完。 | [3.7](./03.07_communication_cases.ipynb) |
| **3.8 问题定位：按步骤排查与日志阅读** | 判断失败发生在哪一步、读三类日志，并故意写错配置练一次修复。 | [3.8](./03.08_debug_and_verify.ipynb) |
| **3.9 本地 WSL 环境与自建绿色包（选看）** | 进阶：装 WSL、装 CANN、编译 HCCL-VM、改码重编、自己打包。 | [3.9](./03.09_appendix_local_env.ipynb) |

**本章课程请从 [3.2 平台背景与原理](./03.02_hccl_vm_platform_overview.ipynb) 开始学习。**

## 两条路径

本章准备了两种环境，选一条走即可：

| | 云端（主线） | 本地（选看，见 3.9） |
| :--- | :--- | :--- |
| 环境 | cannlab 环境（aarch64，自带 CANN 9.0.0） | 自己电脑上的 WSL |
| 要装 CANN | ❌ 不用 | ✅ 要（约 20 分钟） |
| 要编译 | ❌ 不用 | ✅ 要（约 30 分钟） |
| 到出结果 | 约 5 分钟 | 约 1 小时 |
| 能做什么 | 跑测试、改配置、读日志、定位问题 | 还能改源码、改校验规则、重新打包 |
| 适合谁 | 上课、演示、快速验证 | 想深入原理、想自己造包 |

**本地这条路径的全部内容都在 3.9 里讲**（装 WSL、装 CANN、编译、改码、打包）。

**绿色包就是本地"造"出来、给云端"用"的**：把 HCCL-VM 和它需要的运行库预先编好、打成一个压缩包，
学生解压即用。原理见 [3.2](./03.02_hccl_vm_platform_overview.ipynb) 的最后一节。

## 课前准备：把课件放进 Notebook

把课件放进 cannlab 环境有两种方式：

**方式 A：从仓库 clone（推荐）**

在 JupyterLab 里开一个 Terminal，执行：

```bash
git clone -b docs/hccl_notebook_course --depth 1 https://gitcode.com/weixin_45130366/cann-learning-hub_2612.git
ls cann-learning-hub_2612
```

然后在左侧文件列表里双击 `cann-learning-hub_2612/tutorials/hccl_development/03_hccl_vm_guide/03.01_chapter_intro.ipynb`。

**方式 B：手动上传**

把需要的 `.ipynb` 直接拖进左侧文件列表，双击打开。

| | 方式 A（clone） | 方式 B（上传） |
| :--- | :--- | :--- |
| 拿到全部课件 | ✅ 一次到位 | ❌ 要一个个拖 |
| 后续更新 | `git pull` 即可 | 重新拖一遍 |

> 上面的地址指向课程当前所在分支（合入主仓前的过渡地址）；课程合入主仓后也可直接
> `git clone --depth 1 https://gitcode.com/cann/cann-learning-hub.git`。
>
> 如果是私有仓库，clone 需要带 token：
> `git clone https://<用户名>:<token>@gitcode.com/<OWNER>/<REPO>.git`

**绿色包不在仓库里。** 它有 82 MB，放在仓库的 Release 附件中，由 3.3 的下载单元自动获取——
所以 clone 完课件还要跑一遍 3.3。

## 学习目标

学完本章，你应当能够：

- 说清 HCCL-VM 的分层架构，以及 Runner 与 Checker 各自负责什么；
- 在 cannlab 环境里获取绿色包并校验完整性，说清"为什么不需要装 CANN"；
- 认识测试用例：按算子挑选测试程序，会读参数与结果；
- 独立启动模拟环境、选择通信参与者、运行一个通信测试；
- 区分测试用例的数值结论与 Checker 的校验结论，说清模拟环境里为什么数值校验会失败；
- 根据算子、数据规模和参与者分布，配置并组织一组测试；
- 从失败日志中定位问题，改完配置后用可量化的字段验证修复。

## 术语与缩略语

| 概念 | 说明 |
| :--- | :--- |
| HCCL-VM / 北极星 | 在普通 CPU 环境中模拟通信执行并分析任务的工具 |
| 绿色包 | 预先编译好的 HCCL-VM + 测试用例 + 运行库，解压即用，不需要装 CANN |
| CANN | 提供编译与运行所需的接口、库、工具链及算子包 |
| HCCL | 提供 AllReduce、ReduceScatter 等集合通信算子的库 |
| HCOMM | 通信基础库；HCCL-VM 的源码位于 hcomm 仓库的 `test/hccl_vm` |
| 测试用例 / hccl_test | 准备测试数据、调用通信算子并校验结果的测试程序 |
| warmup | 正式计时前的预热迭代，让首次执行的开销不影响测量 |
| 数值校验 | 测试用例把计算结果与预期值比较后给出的结论（模拟环境里通常为 failed） |
| MPI / OpenMPI | MPI 是进程间通信接口标准；用 OpenMPI 启动和协调多个测试进程 |
| rank | 通信参与者的逻辑编号，入门示例用 0、1 两个 rank |
| 通信域 | 本次共同参与集合通信的一组成员 |
| 集群拓扑 | 描述模拟环境中有哪些设备、设备之间怎样连接 |
| Runner | 模拟执行通信任务中的数据操作 |
| Checker | 分析任务记录与依赖关系，检查语义、内存访问与同步 |
| Dump | 把运行现场记录下来（任务、依赖、内存访问）供事后分析 |
| 会话（`(hvm)$>`） | `hccl-vm start` 打开的常驻服务，后续命令都在其中执行 |
| WSL 2 | Windows 提供的 Linux 运行环境，3.9 里用它运行 Ubuntu |

## 学完之后

本章跑通的那条链路——**启动环境 → 建通信域 → 跑测试用例 → Checker 校验**——是后面所有章节的基础。
第 4 章往后会打开 HCCL-VM 的内部：软件架构、拓扑表达、内存与并发、同步与通道，
以及 AICPU、CCU、AIV 三种引擎如何各自执行同一个 AllGather。